# 03 · Full spectrum vs leaflet Fig. 4

The coupled-scan model `braggsim.scan.expected_rate` is calibrated against the spectrum measured in
LD Physics Leaflet P6.3.3.1, Fig. 4 (NaCl, Mo tube, U = 35 kV, I = 1 mA, Δβ = 0.1°, Δt = 10 s),
digitized by `model/scripts/digitize_fig4.py` into `model/data/fig4_digitized.csv`.

We fit $\log_{10} R$ on the log panel, which covers four decades. That way the weak continuum and the
3rd-order lines count as much as the strong 1st-order peaks. The dead time τ is **fixed** at 100 µs: a
typical value for an end-window GM tube, which Fig. 4 cannot pin down (see below).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xraylib
from scipy.optimize import least_squares

from braggsim.constants import D_NACL_PM, MO_KA_PM, MO_KB_PM
from braggsim.crystal import theta_from_lambda
from braggsim.scan import DEFAULT, expected_rate

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
DATA, MODEL, ALT = "#1a1a19", "#2a78d6", "#eb6834"

fig4 = pd.read_csv("../data/fig4_digitized.csv", comment="#")
log_panel = fig4[fig4.panel == "log"]
b, y = log_panel.beta_deg.to_numpy(), np.log10(log_panel.rate_per_s.to_numpy())

## The fit

The fit frees nine physics parameters plus one **calibration-only** offset. The digitized peaks sit a
constant ≈ 0.09° above the Bragg angles, yet the leaflet's own Table 3 lists the measured angles *at* the Bragg
angles. So the shift comes from how the figure was drawn, not from the apparatus. The offset is fitted here to
line up the figure, and `expected_rate` never uses it. Parameters are fitted in log space, which keeps them
positive.

In [ ]:
FREE = ["scale", "line_to_cont", "r2", "r3", "sigma_deg", "leak_amp_per_s", "leak_width_deg",
        "absorber_mg_cm2", "scatter_per_s"]  # fmt: skip
START = DEFAULT.replace(scale=1e5, line_to_cont=0.01, r2=0.5, r3=0.25, sigma_deg=0.1,
                        leak_amp_per_s=1e5, leak_width_deg=1.0, absorber_mg_cm2=200.0,
                        scatter_per_s=5.0)  # fmt: skip


def fit(start, free):
    """Fit log10 R. Returns params, β offset, 1σ errors (relative; offset in °), RMS."""

    def unpack(x):
        return start.replace(**dict(zip(free, np.exp(x[:-1]), strict=True))), x[-1]

    def residual(x):
        p, offset = unpack(x)
        return np.log10(expected_rate(b - offset, 35.0, 1.0, p)) - y

    r = least_squares(residual, np.r_[np.log([getattr(start, k) for k in free]), 0.05])
    cov = np.linalg.pinv(r.jac.T @ r.jac) * np.mean(r.fun**2)
    return *unpack(r.x), np.sqrt(np.diag(cov)), np.sqrt(np.mean(r.fun**2))


params, offset, err, rms = fit(START, FREE)
print(f"log-RMS = {rms:.4f} dex    fig4 β offset = {offset:.4f} ± {err[-1]:.4f}°\n")
print(f"{'parameter':18s} {'fitted':>11s} {'±':>6s} {'DEFAULT':>11s}")
for k, e in zip(FREE, err, strict=False):
    print(f"{k:18s} {getattr(params, k):11.4g} {100 * e:5.1f}% {getattr(DEFAULT, k):11.4g}")

`DEFAULT` in `scan.py` holds these values, rounded, and `model/PARAMETERS.md` documents them.

## Why the model needs an absorber

Without the effective absorber (glass tube wall + anode self-absorption), Kramers' law times the
Jacobian and the Lorentz-polarization factor falls far too slowly with β. The fit then has to distort
everything else to compensate: r₂ drops well below the structure-factor expectation, and the continuum
shape is still wrong.

In [ ]:
no_abs, off0, _, rms0 = fit(START.replace(absorber_mg_cm2=0.0),
                           [k for k in FREE if k != "absorber_mg_cm2"])  # fmt: skip
print(f"without absorber: log-RMS = {rms0:.3f} dex, r2 = {no_abs.r2:.3f}, r3 = {no_abs.r3:.3f}")
print(f"with absorber:    log-RMS = {rms:.3f} dex, r2 = {params.r2:.3f}, r3 = {params.r3:.3f}")

## Model vs Fig. 4 (log and linear panels)

In [ ]:
fine = np.arange(2.0, 25.0001, 0.01)
curves = [("model (DEFAULT)", expected_rate(fine, 35.0, 1.0), MODEL, "-"),
          ("no absorber", expected_rate(fine, 35.0, 1.0, no_abs), ALT, "--")]  # fmt: skip
fig, axes = plt.subplots(3, 1, figsize=(9, 9.5), sharex=True,
                         gridspec_kw={"height_ratios": [3, 3, 1.2]})  # fmt: skip
for ax, panel, scale in ((axes[0], "log", "log"), (axes[1], "linear", "linear")):
    s = fig4[fig4.panel == panel]
    ax.plot(s.beta_deg, s.rate_per_s, ".", ms=3, color=DATA, label=f"Fig. 4, {panel} panel")
    for label, r, color, ls in curves:
        ax.plot(fine + offset, r, ls, color=color, label=label)
    ax.set(yscale=scale, ylabel="R (counts/s)")
    ax.legend(frameon=False)
resid = np.log10(expected_rate(b - offset, 35.0, 1.0) / log_panel.rate_per_s)
axes[2].plot(b, resid, ".", ms=3, color=MODEL)
axes[2].axhline(0, color="#8a8a8a", lw=0.8)
axes[2].set(xlabel="β (°)", ylabel="log₁₀ model/data")
plt.show()

## Peak heights

With a Gaussian instrument profile, the fit matches the line *wings* on the log panel, and the tips of the 1st-
and 2nd-order lines come out 15–30 % lower than in the figure (3rd-order Kα matches). Freeing the Kβ/Kα ratio, giving
the lines their own width, or adding a Lorentzian share (pseudo-Voigt) did not remove it: the log-RMS changes by
< 0.001 dex. Instead of adding a free "peak sharpness" term, we record this as a known deviation.

In [ ]:
rows = []
for n in (1, 2, 3):
    for name, lam in (("Kβ", MO_KB_PM), ("Kα", MO_KA_PM)):
        near = np.abs(b - offset - theta_from_lambda(lam, n)) < 0.2
        data = log_panel.rate_per_s.to_numpy()[near].max()
        model = expected_rate(b[near] - offset, 35.0, 1.0).max()
        rows.append((f"{name} n={n}", data, model, model / data))
print(pd.DataFrame(rows, columns=["line", "Fig. 4 (/s)", "model (/s)", "model/data"])
      .to_string(index=False, float_format="{:.3g}".format))  # fmt: skip

## Plausibility of the fitted values

- **Order reflectivities.** In the kinematic picture, the NaCl (h00) reflections with even h have
  F = 4(f_Na + f_Cl), at sin θ/λ = n/(2d), and r_n ≈ |F_n|²/|F_1|² (LP is already in the model). Thermal
  motion (Debye–Waller, B_Na ≈ 1.7 Å², B_Cl ≈ 1.2 Å² at room temperature) lowers the higher orders;
  extinction, which weakens the strong 1st order most, raises them again. So the fitted r₂ and r₃ are
  *effective* values: they should land between the two estimates, not on either.
- **Absorber.** The fitted areal density, read as borosilicate glass (ρ = 2.23 g/cm³), is a
  ≈ 1.4 mm wall: the right size for the glass envelope of a tube.
- **Width.** σ ≈ 0.12° (FWHM ≈ 0.28°): a few tenths of a degree, as expected for slit collimation.

In [ ]:
q = [n / (2 * D_NACL_PM / 100) for n in (1, 2, 3)]  # sin θ / λ in 1/Å
ff = xraylib.FF_Rayl  # atomic form factor f(Z, sin θ/λ in 1/Å)
for label, b_na, b_cl in (
    ("kinematic, no Debye–Waller", 0.0, 0.0),
    ("with Debye–Waller", 1.7, 1.2),
):
    f = [ff(11, s) * np.exp(-b_na * s**2) + ff(17, s) * np.exp(-b_cl * s**2) for s in q]
    print(f"{label:27s} r2, r3 = {(f[1] / f[0]) ** 2:.3f}, {(f[2] / f[0]) ** 2:.3f}")
print(f"{'fitted':27s} r2, r3 = {params.r2:.3f}, {params.r3:.3f}")
print(f"absorber: {params.absorber_mg_cm2 / 2230 * 10:.2f} mm of glass")
print(f"FWHM = {2.3548 * params.sigma_deg:.3f}°")

## What Fig. 4 cannot tell us

The log-RMS is nearly flat for τ between 0 and 100 µs: the other parameters absorb it. The predicted peak
heights are not flat: at τ = 100 µs the 1st-order Kα tip loses ≈ 25 % to dead time. We keep τ fixed at 100 µs,
and `model/PARAMETERS.md` lists it as a literature value to verify against the LD 559 01 counter. Other quantities stay fixed and are not fitted:
- the line-excitation exponent m = 1.67, which needs scans at several voltages;
- the Kβ/Kα ratio and the doublet structure, from xraylib;
- the counter efficiency ε(λ), which would be degenerate with the absorber.